In [180]:
import pandas as pd

In [181]:
gdp = pd.read_csv(r"C:\Users\devin\Downloads\gdp_data.csv")
meta = pd.read_csv(r"C:\Users\devin\Downloads\country_codes.csv")
military = pd.read_csv(r"C:\Users\devin\Unified-Military-Analytics\military_cleaned.csv")

In [182]:
print("GDP")
print(gdp.shape)
print(gdp.columns.tolist())
print(gdp.head())

print("\n\nMetadata")
print(meta.shape)
print(meta.columns.tolist())
print(meta.head())

print("\n\Military")
print(military.shape)
military.head()

GDP
(13365, 4)
['country_name', 'country_code', 'year', 'value']
  country_name country_code  year        value
0  Afghanistan          AFG  1960  537777811.1
1  Afghanistan          AFG  1961  548888895.6
2  Afghanistan          AFG  1962  546666677.8
3  Afghanistan          AFG  1963  751111191.1
4  Afghanistan          AFG  1964  800000044.4


Metadata
(217, 3)
['country_code', 'region', 'income_group']
  country_code                     region         income_group
0          ABW  Latin America & Caribbean          High income
1          AFG                 South Asia           Low income
2          AGO         Sub-Saharan Africa  Lower middle income
3          ALB      Europe & Central Asia  Upper middle income
4          AND      Europe & Central Asia          High income

\Military
(145, 57)


,country,global_firepower_rank,power_index,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
0,Afghanistan,121,2.7342,40121552,15647405,8826741,842553,75000,0,90000,...,80200000,80200000,49554000000,767000,503000,66000000,652230,0,5987,1200
1,Albania,77,1.7258,3107100,1522479,1292554,62142,7500,0,500,...,50623000,50623000,5692000000,473000,255000,522000000,28748,362,691,41
2,Algeria,27,0.4849,47022473,22570787,19185169,752360,130000,150000,150000,...,100726000000,47963000000,4504000000000,0,3000,233000000,2381740,998,6734,0
3,Angola,59,1.1045,37202061,7440412,3720206,372021,107000,0,10000,...,5514000000,1397000000,343002000000,0,0,0,1246700,1600,5369,1300
4,Argentina,32,0.5983,46994384,20677529,17575900,704916,108000,12370,20000,...,43280000000,46228000000,396464000000,869000,2534000,799999000,2780400,4989,11968,11000


In [183]:
gdp = gdp.rename(
    columns={
        "country_name": "country",
        "value": "gdp_usd"
    }
)

gdp = (
    gdp.sort_values("year")
       .groupby("country", as_index=False)
       .last()
)

gdp = gdp[
    [
        "country",
        "country_code",
        "gdp_usd"
    ]
]

In [184]:
gdp = gdp.merge(
    meta,
    on="country_code",
    how="left"
)

print(gdp.shape)

(262, 5)


In [185]:
mapping = {
    "Egypt, Arab Rep.":"Egypt",
    "Iran, Islamic Rep.":"Iran",
    "Russian Federation":"Russia",
    "Korea, Dem. People's Rep.":"North Korea",
    "Korea, Rep.":"South Korea",
    "Turkiye":"Turkey",
    "Lao PDR":"Laos",
    "Kyrgyz Republic":"Kyrgyzstan",
    "Slovak Republic":"Slovakia",
    "Venezuela, RB":"Venezuela",
    "Yemen, Rep.":"Yemen",
    "Syrian Arab Republic":"Syria",
    "Congo, Dem. Rep.":"Democratic Republic of the Congo",
    "Congo, Rep.":"Republic of the Congo",
    "Taiwan, China":"Taiwan",
    "Cote d'Ivoire":"Ivory Coast"
}

gdp["country"] = gdp["country"].replace(mapping)

In [186]:
military["country"] = military["country"].replace({
    "Beliz": "Belize",
    "Turkiye": "Turkey"
})

In [187]:
df = military.merge(
    gdp,
    on="country",
    how="left"
)

print(df.shape)
print(df.head())

(145, 61)
       country  global_firepower_rank  power_index  total_population  \
0  Afghanistan                    121       2.7342          40121552   
1      Albania                     77       1.7258           3107100   
2      Algeria                     27       0.4849          47022473   
3       Angola                     59       1.1045          37202061   
4    Argentina                     32       0.5983          46994384   

   total_military_manpower  fit_for_service  \
0                 15647405          8826741   
1                  1522479          1292554   
2                 22570787         19185169   
3                  7440412          3720206   
4                 20677529         17575900   

   population_reaching_military_age_annually  active_personnel  \
0                                     842553             75000   
1                                      62142              7500   
2                                     752360            130000   
3         

In [188]:
continent_map = {
    "East Asia & Pacific": "Asia",
    "Europe & Central Asia": "Europe",
    "Latin America & Caribbean": "North America",
    "Middle East & North Africa": "Asia",
    "North America": "North America",
    "South Asia": "Asia",
    "Sub-Saharan Africa": "Africa"
}

df["continent"] = df["region"].map(continent_map)

In [189]:
print(df[df["gdp_usd"].isna()]["country"].tolist())

['North Korea', 'Taiwan']


In [190]:
print(gdp[gdp["country"].str.contains("Tai", case=False, na=False)]["country"])

print(gdp[gdp["country"].str.contains("Korea", case=False, na=False)]["country"])

Series([], Name: country, dtype: object)
123    South Korea
Name: country, dtype: object


In [191]:
gdp = gdp.drop_duplicates(subset="country", keep="first")

## KPI Engineering

In [192]:
df["gdp_rank"] = (
    df["gdp_usd"]
    .rank(method="dense", ascending=False)
)

In [193]:
asset_cols = [
    "tanks",
    "armored_fighting_vehicles",
    "self_propelled_artillery",
    "towed_artillery",
    "rocket_projectors",
    "fighter_aircraft",
    "attack_aircraft",
    "transport_aircraft",
    "trainer_aircraft",
    "special_mission_aircraft",
    "tanker_aircraft",
    "attack_helicopters",
    "submarines",
    "destroyers",
    "frigates",
    "corvettes",
    "aircraft_carriers",
    "helicopter_carriers"
]

df["total_assets"] = df[asset_cols].fillna(0).sum(axis=1)

df["assets_per_capita"] = (
    df["total_assets"] /
    df["total_population"]
)

In [194]:
df["budget_to_gdp_ratio"] = (
    df["defense_budget_usd"] /
    df["gdp_usd"]
) * 100

In [195]:
df["power_index_rank_gap"] = (
    df["gdp_rank"] -
    df["global_firepower_rank"]
)

In [196]:
nato = {
    "Albania","Belgium","Bulgaria","Canada","Croatia",
    "Czech Republic","Denmark","Estonia","Finland",
    "France","Germany","Greece","Hungary","Iceland",
    "Italy","Latvia","Lithuania","Luxembourg",
    "Montenegro","Netherlands","North Macedonia",
    "Norway","Poland","Portugal","Romania",
    "Slovakia","Slovenia","Spain","Sweden",
    "Turkey","United Kingdom","United States"
}

df["is_nato"] = df["country"].isin(nato)

In [197]:
id_columns = [
    "country",
    "country_code",
    "region",
    "continent",
    "income_group",
    "is_nato"
]

long_df = df.melt(
    id_vars=id_columns,
    var_name="metric",
    value_name="value"
)

long_df.to_excel(
    r"C:\Users\devin\Unified-Military-Analytics\military_long.xlsx",
    index=False
)

In [198]:
df.to_excel(r"C:\Users\devin\Unified-Military-Analytics\military_final.xlsx",index=False)

print("Module 3 Completed Successfully!")
print(df.shape)

print("\nMissing Percentage")
print(
    (df.isna().mean()*100)
    .sort_values(ascending=False)
    .head(10)
)

Module 3 Completed Successfully!
(145, 68)

Missing Percentage
power_index_rank_gap    1.37931
country_code            1.37931
region                  1.37931
gdp_usd                 1.37931
budget_to_gdp_ratio     1.37931
gdp_rank                1.37931
continent               1.37931
income_group            1.37931
total_population        0.00000
power_index             0.00000
dtype: float64
